# Calculating quadratic density $\eta(p_k)$ from $\Delta H(p)$
In notebook "13_DelHdisplay" we have seen that the populations of gaps within the interval of survival
$\Delta H(p_k)$ are an approximately fair sampling from the relative populations at that stage of Eratosthenes sieve.

The length of the interval $\Delta H(p_k)$ is proportional to the gap $g= p_{k+1}-p_k$.
$$ | \Delta H(p_k) | = p_{k+1}^2 - p_k^2 = g \cdot (2\cdot p_k+g)$$
We observe this proportionality to $g$ in those histograms of the populations of gaps within $\Delta H$.  We color-coded
the band for $\Delta H(p_k)$ by the value of $g=p_{k+1}-p_k$.

Continuing our studies of $\Delta H(p_k)$, here we study the quadratic density $\eta(p)$ of the intervals of survival.
$$ \eta(p_k) = \frac{1}{p_{k+1}-p_k} |N_{\Delta H}(p_k)|$$
This gives us the average population of a gap $g$ in an interval $[n^2,(n+1)^2]$ in $\Delta H(p_k)$.  

This provides indirect support for Legendre's conjecture that there is always a prime in the interval $[n^2, (n+1)^2]$.
In fact, given the relative populations of gaps, we expect an abundance of gaps in this interval.  We see below that the
samples across consecutive intervals of survival $\Delta H(p)$ display low variance.

In [1]:
%reset -f

import pandas as pd
import numpy as np
from numpy.polynomial.polynomial import polyval
import array
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
#from plotnine import ggplot, geom_point, aes, geom_line, theme, ggsave
import gc
import psutil
import sys
import pickle

import itertools
from ipywidgets import interact
import ipywidgets as widgets
from IPython.display import display
plt.ion


<function matplotlib.pyplot.ion() -> 'AbstractContextManager'>

In [2]:
try:
    smallprimes = np.load('primesE9.npy')
except FileNotFoundError:
    print("File primesE9.npy not found.  See 01 notebooks.")
    sys.exit(1)



In [3]:
# block to check the available system memory
gc.collect()
memory = psutil.virtual_memory()
available_memory = memory.available
del memory
print(f"Available memory: {available_memory / (1024 ** 2):.2f} MB")

Available memory: 4358.94 MB


In [4]:
lenp = len(smallprimes)
maxp = smallprimes[-1]
print(f"Length primes {lenp} primes {smallprimes[0:5]}...{smallprimes[10]} maxp {maxp}")


Length primes 51961884 primes [ 3  5  7 11 13]...37 maxp 1023101273


In [5]:
lambda_arr = np.zeros(lenp)

In [6]:
# smallprimes[0:5] are 3,5,7,11,13,
lambda_arr[0] = 1
ilam = 0
ip = 10 # the offset from lambda_arr into smallprimes for p0=37
while (ip < (lenp-1)):
    ilam += 1
    ip += 1
    lambda_arr[ilam] = lambda_arr[ilam-1]*(smallprimes[ip]-3)/(smallprimes[ip]-2)

In [7]:
lambda_arr[-100:-1] # The last nine entries in the array are 0

array([0.17990707, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990706, 0.17990706,
       0.17990706, 0.17990706, 0.17990706, 0.17990705, 0.17990705,
       0.17990705, 0.17990705, 0.17990705, 0.17990705, 0.17990705,
       0.17990705, 0.17990705, 0.17990705, 0.17990705, 0.17990705,
       0.17990705, 0.17990705, 0.17990705, 0.17990705, 0.17990

## Accumulations over intervals $\Delta H(p)$
For comparison with the relative populations $w_{g,1}(p^\#)$, we accumulate the counts of gaps within the intervals of survival $\Delta H(p) = [p^2, q^2]$.

## Quadratic density $\eta(p_k)$
From the counts of gaps in the intervals of survival $\Delta H(p_k)$ we calculate the quadratic residues
$$\eta(p_k) = \frac{1}{p_{k+1}-p_k} \cdot N_{\Delta H}(p_k).$$
As $p_k$ grows the length of the interval $\Delta H(p_k)$ grows almost proportionally to the gap $g=p_{k+1}-p_k $.
$$|\Delta H(p_k)| = p_{k+1}^2- p_k^2 = g \cdot(2p_k+g)$$


In [24]:
# Develop a master color dictionary - up to g=100
# we set colors by family, as determined by prime factors of the gap
mastercolordict={'2':'#FF0000', '4':'#FF8888', '8':'#EEBBBB', '16':'#FF99CC', '32':'#FFDDDD', '64':'#FFEEEE',
                 '6':'#0000FF', '12':'#00DDFF', '18':'#6666FF', '24':'#BBBBFF', '36':'#DDDDFF', '48':'#0000AF',
                 '54':'#44448F', '72':'#88888F',
                 '10':'#00CC00', '20':'#98FB98', '40':'#DDFFCF', '50':'#BBFFBB', '80':'#44AA66', '100':'#228F3F',
                 '30':'#FFD700', '60':'#FFB000', '90':'#FFCC33',
                 '14':'#DD00EE', '28':'#AA22CC', '56':'#660066', '98':'#662266',
                 '42':'#0088BB', '84':'#4488BB', '70':'#00BB88', 
                 '22':'#884400', '44':'#884444', '66':'#884488', '88':'#AA6644',
                 '26':'#AAAAAA', '52':'#888888', '78':'#6666AA',
                 '34':'#CC0000', '68':'#CC4444', '38':'#AA8800', '46':'#004488', '58':'#448800', '76':'#AA9944'}

In [30]:
# global variables describing the data for the figures
prevsamplename = 'Exx_nnn'
min_samp = 0  # low index for DelH in data -- not from selection range
max_samp = 5  # high index for DelH in data -- not from selection range
num_DH = 5
prevlowsamp = 0  # low index for DelH from selection range
prevhighsamp = 60 # high index for DelH from selection range
mingap = 2 # The gap g is an even number with index (g-2)/2
maxgap = 120  # Initial value. Max gap value for widget
maxlegend = 500
gapnames = np.arange(2,240, 2, dtype=int)


In [31]:
lowgapdex = 0
highgapdex = 45  # the gap index is half the gap size:  gap = 2*(gapdex+1)

def draw_eta(samplename, gaprange, DHrange):   # the input parameters are indices, not the prime values themselves
    global min_samp
    global max_samp
    global num_DH
    global prevlowsamp
    global prevhighsamp
    global mingap
    global maxgap
    global maxlegend
    global prevsamplename

    samplefile = 'DH'+samplename
    with open(samplefile,"rb") as fp2:
        DHff = pickle.load(fp2)
    fp2.close()

    plt.ioff()  # turn interactive mode off
    
    # unpack the data sample:  first large prime in DH, index & first prime for DelH(p), and counts in DelH(nDH, gaps)
    minP = DHff[0]   # start of the sampled primes beyond p^2
    iminp = DHff[1][0]
    minp = DHff[1][1]
    DelH = DHff[2]

    num_DH = DelH.shape[0]
    maxgapdex = DelH.shape[1]
    maxgap = 2*maxgapdex+2
    max_samp = num_DH-1

    if (samplename != prevsamplename):
        prevsamplename = samplename

        # reset slider values for new file
        xDHSelect.max = num_DH
        xDHSelect.value = [0,min(num_DH,maxlegend)]
        xgapsSelect.value = [2,92]

        lowgapdex = 0
        highgapdex = 45

        lowDHdex = 0
        highDHdex = min(num_DH, maxlegend)
    else:
        # the gap index is half the gap size:  gap = 2*(gapdex+1)
        lowgapdex = int((gaprange[0]-2)/2)
        highgapdex = int((gaprange[1]+2)/2)

        # set DH indices
        lowDHdex = DHrange[0]
        highDHdex = DHrange[1]
        # check values against bounds
        if ((num_DH > maxlegend) and (highDHdex - lowDHdex > maxlegend)):
            if (highDHdex != prevhighsamp):
                lowDHdex = highDHdex - maxlegend
            else:
                highDHdex = lowDHdex + maxlegend
            xDHSelect.value = [lowDHdex, highDHdex]
            
        prevlowsamp = lowDHdex
        prevhighsamp = highDHdex        

    print(f"{num_DH} DelH from {minp} to {smallprimes[iminp+num_DH]} gaps to {maxgap}")

    gaplabels = (gapnames[lowgapdex:highgapdex]).astype(str)
    numgaps = highgapdex-lowgapdex

    lowpdex = iminp+lowDHdex
    highpdex = iminp + highDHdex
    primerange = (smallprimes[lowpdex:highpdex]).astype(str)
    
    # adjust indices for DelH sample and copy this over for the eta-sample
    print(f"pE9 {lowpdex}:{highpdex+1} DelH {lowpdex-iminp}:{highpdex-iminp}")
    partialeta = DelH[lowDHdex:highDHdex, lowgapdex:highgapdex].copy()

    numprimes = np.sum(partialeta[:,:])
    
    # calculate quadratic densities eta
    ip = lowpdex
    iDH = 0
    while (ip < highpdex):
        gap2 = smallprimes[ip+1]-smallprimes[ip]
        jg = 0
        while (jg < numgaps):
            partialeta[iDH,jg] /= gap2
            jg += 1
        ip += 1
        iDH += 1

    partialeta = partialeta.transpose()
    
    flat_eta = partialeta.flatten()
    xgaps = np.repeat(gaplabels, (highpdex-lowpdex))

    # create the color dictionary, using gapsizes in primesE9 from lowpdex to highpdex
    i = lowgapdex
    colordict = []
    while (i < highgapdex):
        j=lowpdex
        while (j < highpdex):
            pstring = str(smallprimes[j])
            gapstring = str((smallprimes[j+1]-smallprimes[j]))
            if gapstring in mastercolordict:
                colorstring = mastercolordict[gapstring]
            else:
                print(f"Missing color for {gapstring} at {pstring}")
                colorstring = '#080808'
            colordict.append(colorstring)
            j += 1
        i += 1

    fig, ax = plt.subplots()
    fig.set_size_inches(12,8)

    lamvalue0 = lambda_arr[lowpdex-10]
    lamvalue1 = lambda_arr[highpdex-10]
    middex = int((highpdex+lowpdex)/2)
    ptitle = (smallprimes[middex])**2

    ax.set_title(f"Quadratic densities over {numprimes:,} prime gaps in {num_DH} intervals of survival around {ptitle:.3e}\n primes {smallprimes[lowpdex]}-{smallprimes[highpdex]},  $\lambda \in$ [{lamvalue1:.3f},{lamvalue0:.3f}]")
    plt.grid(True,axis='y',color='#A0A0A0',lw=0.4)

    eta_mu = np.zeros(numgaps)
    eta_std = np.zeros(numgaps)
    i=0
    while (i < numgaps):
        eta_mu[i] = np.mean(partialeta[i,:])
        eta_std[i] = np.std(partialeta[i,:])
        i += 1

    eta_stdmark0 = eta_mu - eta_std
    eta_stdmark00 = eta_mu - 2*eta_std
    eta_stdmark1 = eta_mu + eta_std
    eta_stdmark11 = eta_mu + 2*eta_std

    plt.scatter(gaplabels,eta_mu, marker='D', color='black', s=90)
    plt.scatter(gaplabels,eta_stdmark0, marker='_', color='black', s=150)
    plt.scatter(gaplabels,eta_stdmark1, marker='_', color='black', s=150)
    plt.scatter(gaplabels,eta_stdmark00, marker='_', color='black', s=150)
    plt.scatter(gaplabels,eta_stdmark11, marker='_', color='black', s=150)
    plt.scatter(xgaps,flat_eta, c=colordict, marker='x', s=25)

    plt.show()

# Interactive controls
xsampleSelect = widgets.SelectionSlider(value='E09_900', options=[('0.220','E14_568'),('0.252','E12_742'),('0.275','E11_640'),
                                                 ('0.325','E10_108'),('0.329','E09_900'),('0.346','E09_360')], disabled=False, description='lambda(p)', layout=widgets.Layout(width='50%'))
xgapsSelect = widgets.IntRangeSlider(value=[2,90],min=2, max=maxgap, step=2, 
                  description="gaps shown", layout=widgets.Layout(width='80%'), disabled=False)
xDHSelect = widgets.IntRangeSlider(value=[0,int((num_DH+1)/2)], min=0, max=num_DH, step=1, 
                  description="DelH shown", layout=widgets.Layout(width='80%'), disabled=False)

# save figure checkbox... XXXQHERE

interact(draw_eta, samplename=xsampleSelect, gaprange=xgapsSelect, DHrange=xDHSelect)


interactive(children=(SelectionSlider(description='lambda(p)', index=4, layout=Layout(width='50%'), options=((…

<function __main__.draw_eta(samplename, gaprange, DHrange)>

## Quadratic density for intervals of survival $\Delta H(p)$
The interactive figure above displays the quadratic density of prime gaps by size of gap, across consecutive intervals of survival $\Delta H(p_k) = [p_k^2, p_{k+1}^2]$.

We color-code the marker for $\Delta H(p_k)$ by the size of the gap $p_{k+1}-p_k$.  For each gap we also show the mean of the samples and two standard deviations above and below the mean.
